### Sentiment Analysis

Sentiment analysis, also known as opinion mining, is a natural language processing (NLP) technique used to determine the emotional tone behind a body of text. It is commonly used to identify and categorize opinions expressed in text as positive, negative, or neutral.

#### Applications of Sentiment Analysis:
- **Customer Feedback Analysis**: Understanding customer opinions about products or services.
- **Social Media Monitoring**: Analyzing public sentiment about brands, events, or trends.
- **Market Research**: Gaining insights into consumer behavior and preferences.
- **Reputation Management**: Tracking and managing the sentiment around a company or individual.

#### How It Works:
1. **Text Preprocessing**: The input text is cleaned and tokenized to prepare it for analysis.
2. **Feature Extraction**: Relevant features, such as words, phrases, or patterns, are extracted.
3. **Classification**: The text is classified into sentiment categories (e.g., positive, negative, neutral) using machine learning models or rule-based approaches.

Sentiment analysis is widely used in various industries to make data-driven decisions and improve customer experiences.

GPT-5

In [ ]:
import os
import json
from openai import OpenAI
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time
from datetime import datetime

# It's best practice to set your API key as an environment variable.
# DO NOT hardcode your key directly in the script.
# You can set it in your terminal like: export OPENAI_API_KEY='your-key-here'
# Or, for testing, you can uncomment and replace the line below.
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

try:
    # Initialize the OpenAI client
    # The client automatically looks for the OPENAI_API_KEY environment variable.
    client = OpenAI()
except Exception as e:
    print(f"Error initializing OpenAI client: {e}")
    print("Please make sure the OPENAI_API_KEY environment variable is set.")
    exit()

def get_sentiment(text_to_analyze: str, model_to_use: str) -> dict:
    """
    Calls the OpenAI Responses API with the specified model to analyze sentiment and emotion.
    
    Args:
        text_to_analyze: The text string to analyze.
        model_to_use: The name of the model to use (e.g., "gpt-5").

    Returns:
        A dictionary containing sentiment and emotion (e.g., 
        {"sentiment": "Positive", "emotion": "joy"}) or an error message.
    """
    
    # This is the developer prompt (similar to a system prompt for gpt-5).
    # It instructs the model on its role and, most importantly,
    # restricts its output format to a specific JSON structure.
    developer_prompt = (
        "You are an expert text analysis assistant. Analyze the sentiment and emotion "
        "of the text provided by the user. "
        "Respond with only a single, valid JSON object in the format: "
        '{"sentiment": "Positive/Negative/Neutral", "emotion": "love/anger/sadness/fear/happy/neutral"}'
        "Only output the JSON object and nothing else."
    )

    try:
        # gpt-5 uses the client.responses.create endpoint
        response = client.responses.create(
            # Use the model name passed into the function
            model=model_to_use,
            
            # The 'input' parameter replaces 'messages'
            input=[
                { "role": "developer", "content": developer_prompt },
                { "role": "user", "content": text_to_analyze }
            ],
            
            # We can control reasoning and verbosity for gpt-5
            # 'minimal' reasoning and 'low' verbosity are best for classification
            reasoning={"effort": "minimal"},
            text={"verbosity": "low"},
        )
        
        # Extract the model's reply from 'output_text'
        response_text = response.output_text.strip()
        
        # Try to parse the model's response as JSON
        try:
            prediction_data = json.loads(response_text)
            # Basic validation
            if "sentiment" in prediction_data and "emotion" in prediction_data:
                return prediction_data
            else:
                print(f"Warning: JSON response missing keys: {response_text}")
                return {"sentiment": "Unknown", "emotion": "Unknown"}
        except json.JSONDecodeError:
            # Handle cases where the model did not return valid JSON
            print(f"Warning: Failed to decode JSON response: {response_text}")
            return {"sentiment": "Unknown", "emotion": "Unknown"}

    except openai.APIConnectionError as e:
        print(f"Error: Could not connect to OpenAI API. {e}")
        return {"sentiment": "Error", "emotion": "Error"}
    except openai.RateLimitError as e:
        print(f"Error: Rate limit exceeded. {e}")
        return {"sentiment": "Error", "emotion": "Error"}
    except openai.APIStatusError as e:
        print(f"Error: OpenAI API returned an error (Status code: {e.status_code}). {e}")
        return {"sentiment": "Error", "emotion": "Error"}
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return {"sentiment": "Error", "emotion": "Error"}

def format_confusion_matrix(cm, labels, title):
    """Formats a confusion matrix into a readable string."""
    matrix_str = f"\n--- Confusion Matrix ({title}) ---\n"
    
    # Create header row (Predicted labels)
    col_width = max(max(len(label) for label in labels), 7) + 2 # Get max label length for padding
    header = "True \\ Pred".ljust(col_width) + " | "
    for label in labels:
        header += f"{label:^{col_width}} | "
    matrix_str += header + "\n"
    matrix_str += "-" * len(header) + "\n"
    
    # Create matrix rows (True labels)
    for i, label in enumerate(labels):
        row_str = f"{label.ljust(col_width)} | "
        for val in cm[i]:
            row_str += f"{str(val):^{col_width}} | "
        matrix_str += row_str + "\n"
    
    return matrix_str + "\n"

def test_model_accuracy(csv_file_path: str, sample_size: int = 50):
    """
    Loads a CSV, runs sentiment/emotion analysis on a sample, and reports accuracy.
    Saves the detailed results to '[model_name]_[timestamp]_results.csv' and
    the classification report to '[model_name]_[timestamp]_report.txt'.
    """
    print(f"Loading dataset from {csv_file_path}...")
    try:
        # Try parsing as a standard CSV first (comma-separated)
        df = pd.read_csv(csv_file_path)
        
    except FileNotFoundError:
        print(f"Error: File not found at '{csv_file_path}'.")
        print("Please make sure the CSV file path is correct.")
        return
    except pd.errors.ParserError as e:
        # This catches errors like 'Error tokenizing data'
        print(f"Standard CSV parsing failed: {e}")
        print("Trying again with tab delimiter...")
        try:
            # Fallback to tab-separated (which the previous file was)
            df = pd.read_csv(csv_file_path, sep='\t')
        except Exception as e2:
            print(f"Tab-parsing failed: {e2}. Trying with 'python' engine...")
            try:
                # Final fallback for complex, messy files
                df = pd.read_csv(csv_file_path, engine='python', on_bad_lines='skip')
            except Exception as e3:
                print(f"Fatal error: Could not parse file even with fallbacks. {e3}")
                return
    except Exception as e:
        # Catch other potential errors (e.g., permissions)
        print(f"General error loading file: {e}")
        return

    # --- IMPORTANT: Adjust these column names based on your CSV ---
    TEXT_COLUMN = "tweet" 
    LABEL_COLUMN = "label"
    # -----------------------------------------------------------
    
    # --- Define the model to use for this test run ---
    # This variable will be used for the API call and the report filenames
    model_name = "gpt-5"
    # -------------------------------------------------

    if TEXT_COLUMN not in df.columns or LABEL_COLUMN not in df.columns:
        print(f"Error: Columns '{TEXT_COLUMN}' or '{LABEL_COLUMN}' not found in the file.")
        print(f"Available columns are: {list(df.columns)}")
        print("This might happen if the delimiter is still incorrect or the file is corrupt.")
        return

    # Limit to a sample to avoid high API costs/time
    if len(df) > sample_size:
        print(f"Using a random sample of {sample_size} tweets from {len(df)} total.")
        df_sample = df.sample(n=sample_size, random_state=42) # random_state for reproducible results
    else:
        print(f"Using all {len(df)} tweets in the file.")
        df_sample = df

    print(f"\n--- Starting Sentiment Analysis ---")
    print(f"Model: {model_name}")
    print(f"WARNING: This will make {len(df_sample)} API calls to OpenAI.")
    print("This may take some time and incur costs.\n")
    
    results_list = [] # Store detailed results for saving to CSV

    # Map Indonesian labels from CSV to the model's English output
    # This is an EMOTION dataset, so we map emotions to sentiment.
    label_mapping = {
        "love": "Positive",
        "happy": "Positive", # <-- Ditambahkan label 'happy'
        "anger": "Negative",
        "sadness": "Negative",
        "fear": "Negative"
        # We ignore any other labels as they don't map to sentiment.
    }

    processed_count = 0
    for index, row in df_sample.iterrows():
        text = str(row[TEXT_COLUMN])
        true_label_raw = str(row[LABEL_COLUMN]).strip() # This is a string (e.g., "love")
        
        # --- Skip rows with empty text or labels ---
        if not text or pd.isna(text):
            print(f"Skipping row {index}: Empty text.")
            continue
        if pd.isna(true_label_raw) or not true_label_raw:
            print(f"Skipping row {index}: Empty label.")
            continue
            
        # Get the mapped sentiment (e.g., "Positive") from the emotion label (e.g., "love")
        true_sentiment = label_mapping.get(true_label_raw)
        
        if true_sentiment is None:
            print(f"Skipping row {index}: Unknown/unmapped true label '{true_label_raw}'.")
            continue
        # ---------------------------------------------

        # Call the API to get the predicted sentiment and emotion
        # Pass the defined model_name to the function
        prediction_data = get_sentiment(text, model_name)
        
        predicted_sentiment = prediction_data.get("sentiment", "Unknown")
        predicted_emotion = prediction_data.get("emotion", "Unknown")
        
        processed_count += 1
        
        # --- Store results for CSV export ---
        is_correct_sentiment = False
        is_correct_emotion = False
        if predicted_sentiment not in ["Error", "Unknown"]:
            is_correct_sentiment = (true_sentiment == predicted_sentiment)
            is_correct_emotion = (true_label_raw == predicted_emotion)
        
        results_list.append({
            "Text": text,
            "True_Emotion": true_label_raw, 
            "True_Sentiment": true_sentiment,
            "Predicted_Emotion": predicted_emotion,
            "Predicted_Sentiment": predicted_sentiment,
            "Correct_Sentiment": is_correct_sentiment,
            "Correct_Emotion": is_correct_emotion
        })
        # -----------------------------------

        if predicted_sentiment not in ["Error", "Unknown"]:
            print(f"\nTweet Text: {text}")
            print(f"Processed tweet {processed_count}/{len(df_sample)}...")
            print(f"  > Sentiment: True: {true_sentiment}, Pred: {predicted_sentiment}")
            print(f"  > Emotion:   True: {true_label_raw}, Pred: {predicted_emotion}")
        else:
            print(f"\nTweet Text: {text}") # Also print text on error/skip
            print(f"Skipping tweet {processed_count}/{len(df_sample)} due to API error: {predicted_sentiment}")

        # IMPORTANT: Add a small delay to avoid hitting API rate limits
        time.sleep(0.5) # 0.5 second delay between requests

    print("\n--- Analysis Complete ---")
    
    # --- Save detailed results to CSV ---
    if not results_list:
        print("No results to report. All rows may have been skipped.")
        return
        
    results_df = pd.DataFrame(results_list)
    
    # --- Generate dynamic file paths ---
    # The model_name variable is now correctly used here
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    results_csv_path = f"{model_name}_{timestamp}_results.csv"
    # -----------------------------------
    
    try:
        results_df.to_csv(results_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSaved detailed results to {results_csv_path}")
    except Exception as e:
        print(f"\nError saving results CSV: {e}")
    # ----------------------------------

    # --- Calculate metrics from valid predictions ---
    valid_results_df = results_df[~results_df['Predicted_Sentiment'].isin(["Error", "Unknown"])]
    
    if valid_results_df.empty:
        print("No valid predictions were made. Cannot generate report.")
        return
        
    # --- 1. SENTIMENT Report ---
    y_true_sentiment = valid_results_df['True_Sentiment']
    y_pred_sentiment = valid_results_df['Predicted_Sentiment']
    all_labels_sentiment = ["Positive", "Negative", "Neutral"]

    accuracy_sentiment = accuracy_score(y_true_sentiment, y_pred_sentiment)
    report_sentiment = classification_report(
        y_true_sentiment, 
        y_pred_sentiment, 
        labels=all_labels_sentiment, 
        zero_division=0
    )
    
    # --- Create Sentiment Confusion Matrix ---
    cm_sentiment = confusion_matrix(y_true_sentiment, y_pred_sentiment, labels=all_labels_sentiment)
    cm_str_sentiment = format_confusion_matrix(cm_sentiment, all_labels_sentiment, "Sentiment")
    
    # --- 2. EMOTION Report ---
    y_true_emotion = valid_results_df['True_Emotion']
    y_pred_emotion = valid_results_df['Predicted_Emotion']
    # Get all unique emotion labels present in the true data for the report
    all_labels_emotion = sorted(list(label_mapping.keys()))

    accuracy_emotion = accuracy_score(y_true_emotion, y_pred_emotion)
    report_emotion = classification_report(
        y_true_emotion, 
        y_pred_emotion, 
        labels=all_labels_emotion, 
        zero_division=0
    )
    
    # --- Create Emotion Confusion Matrix ---
    cm_emotion = confusion_matrix(y_true_emotion, y_pred_emotion, labels=all_labels_emotion)
    cm_str_emotion = format_confusion_matrix(cm_emotion, all_labels_emotion, "Emotion")

    
    # --- Print to Console ---
    print("\n--- SENTIMENT ANALYSIS REPORT ---")
    print(f"Accuracy: {accuracy_sentiment * 100:.2f}%\n")
    print("Classification Report (Sentiment):")
    print(report_sentiment)
    print(cm_str_sentiment) # Print sentiment confusion matrix
    
    print("\n--- EMOTION ANALYSIS REPORT ---")
    print(f"Accuracy: {accuracy_emotion * 100:.2f}%\n")
    print("Classification Report (Emotion):")
    print(report_emotion)
    print(cm_str_emotion) # Print emotion confusion matrix
    
    # --- Save classification report to TXT ---
    # Use the same model and timestamp from above
    report_txt_path = f"{model_name}_{timestamp}_report.txt"
    
    try:
        with open(report_txt_path, "w", encoding="utf-8") as f:
            f.write(f"Sentiment & Emotion Analysis Report\n")
            f.write(f"Source File: {csv_file_path}\n")
            f.write(f"Sample Size: {sample_size} (Processed: {len(results_df)}, Valid API Responses: {len(valid_results_df)})\n")
            f.write(f"Model: {model_name}\n") # Use variable
            
            f.write("\n\n--- SENTIMENT ANALYSIS REPORT ---\n")
            f.write("----------------------------------\n")
            f.write(f"Accuracy: {accuracy_sentiment * 100:.2f}%\n\n")
            f.write("Classification Report (Sentiment):\n")
            f.write(report_sentiment)
            f.write(cm_str_sentiment) # Save sentiment confusion matrix
            
            f.write("\n\n--- EMOTION ANALYSIS REPORT ---\n")
            f.write("----------------------------------\n")
            f.write(f"Accuracy: {accuracy_emotion * 100:.2f}%\n\n")
            f.write("Classification Report (Emotion):\n")
            f.write(report_emotion)
            f.write(cm_str_emotion) # Save emotion confusion matrix
            
        print(f"\nSaved combined classification reports to {report_txt_path}")
    except Exception as e:
        print(f"\nError saving report TXT: {e}")
    # -----------------------------------------


# --- Example Usage ---
if __name__ == "__main__":
    # --- Comment out old examples ---
    # text1 = "Makanannya enak sekali dan pelayanannya ramah!"
    # sentiment1 = get_sentiment(text1, "gpt-5") # Requires model name now
    # print(f"Text: '{text1}'\nSentiment: {sentiment1}\n")

    # text2 = "This movie was incredibly boring. I fell asleep halfway through."
    # sentiment2 = get_sentiment(text2, "gpt-5") # Requires model name now
    # print(f"Text: '{text2}'\nSentiment: {sentiment2}\n")

    # text3 = "The package is scheduled to arrive tomorrow at 4 PM."
    # sentiment3 = get_sentiment(text3, "gpt-5") # Requires model name now
    # print(f"Text: '{text3}'\nSentiment: {sentiment3}\n")

    # text4 = "The food was great, but the music was way too loud."
    # sentiment4 = get_sentiment(text4, "gpt-5") # Requires model name now
    # print(f"Text: '{text4}'\nSentiment: {sentiment4}\n")

    # --- New Accuracy Testing ---
    # IMPORTANT: Place your CSV file in the same folder as this script
    # Make sure this file name is correct.
    # Updated to the path you provided.
    FILE_PATH = "<YOUR_CSV_FILE_PATH_HERE>"
    
    # Start with a small sample (e.g., 50) to test.
    # Running all 10,000+ rows will be slow and cost money.
    # You can increase this number once you confirm it's working.
    SAMPLE_SIZE = 440
    
    test_model_accuracy(FILE_PATH, SAMPLE_SIZE)

GPT-4 and below

### Difference Between Using GPT-5 and GPT-4 in Python

1. **Performance and Accuracy**:
    - GPT-5 generally offers improved performance, better reasoning, and higher accuracy compared to GPT-4.
    - It is more capable of handling complex tasks and understanding nuanced instructions.

2. **API Features**:
    - GPT-5 may include additional features or parameters (e.g., enhanced reasoning controls, verbosity settings) not available in GPT-4.
    - GPT-4 typically uses the `chat.completions` endpoint, while GPT-5 might use a newer or more advanced endpoint like `responses.create`.

3. **Cost and Speed**:
    - GPT-5 is likely to be more expensive per token compared to GPT-4.
    - Depending on the task, GPT-5 may also be faster due to optimizations.

4. **Use Cases**:
    - GPT-4 is sufficient for simpler tasks like basic text generation or classification.
    - GPT-5 is better suited for tasks requiring high precision, such as advanced sentiment analysis or multi-step reasoning.

5. **Backward Compatibility**:
    - GPT-5 may introduce new parameters or response formats, requiring updates to existing code.
    - GPT-4 code can often be reused with minimal changes when upgrading to GPT-5.

In [ ]:
import os
import json # Added import for JSON parsing
from openai import OpenAI
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time
from datetime import datetime

# It's best practice to set your API key as an environment variable.
# DO NOT hardcode your key directly in the script.
# You can set it in your terminal like: export OPENAI_API_KEY='your-key-here'
# Or, for testing, you can uncomment and replace the line below.
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

try:
    # Initialize the OpenAI client
    # The client automatically looks for the OPENAI_API_KEY environment variable.
    client = OpenAI()
except Exception as e:
    print(f"Error initializing OpenAI client: {e}")
    print("Please make sure the OPENAI_API_KEY environment variable is set.")
    exit()

def get_sentiment(text_to_analyze: str, model_to_use: str) -> dict:
    """
    Calls the OpenAI Chat Completions API with the specified model to analyze sentiment and emotion.
    
    Args:
        text_to_analyze: The text string to analyze.
        model_to_use: The name of the model to use (e.g., "gpt-4o").

    Returns:
        A dictionary containing sentiment and emotion (e.g., 
        {"sentiment": "Positive", "emotion": "joy"}) or an error message.
    """
    
    # This is the system prompt.
    # It instructs the model on its role and, most importantly,
    # restricts its output format to a specific JSON structure.
    system_prompt = (
        "You are an expert text analysis assistant. Analyze the sentiment and emotion "
        "of the text provided by the user. "
        "Respond with only a single, valid JSON object in the format: "
        '{"sentiment": "Positive/Negative/Neutral", "emotion": "love/anger/sadness/fear/happy/neutral"}'
        "Only output the JSON object and nothing else."
    )

    try:
        # gpt-4 models use the client.chat.completions.create endpoint
        response = client.chat.completions.create(
            # Use the model name passed into the function
            model=model_to_use,
            
            # Use the 'messages' parameter
            messages=[
                { "role": "system", "content": system_prompt },
                { "role": "user", "content": text_to_analyze }
            ],
            
            # This ensures the model *must* output valid JSON
            response_format={"type": "json_object"},
            
            temperature=0, # Set temperature to 0 for classification tasks
        )
        
        # Extract the model's reply from the message content
        response_text = response.choices[0].message.content.strip()
        
        # Try to parse the model's response as JSON
        # The 'json_object' response_format should guarantee this, but we check anyway.
        try:
            prediction_data = json.loads(response_text)
            # Basic validation
            if "sentiment" in prediction_data and "emotion" in prediction_data:
                return prediction_data
            else:
                print(f"Warning: JSON response missing keys: {response_text}")
                return {"sentiment": "Unknown", "emotion": "Unknown"}
        except json.JSONDecodeError:
            # Handle cases where the model did not return valid JSON
            print(f"Warning: Failed to decode JSON response: {response_text}")
            return {"sentiment": "Unknown", "emotion": "Unknown"}

    except openai.APIConnectionError as e:
        print(f"Error: Could not connect to OpenAI API. {e}")
        return {"sentiment": "Error", "emotion": "Error"}
    except openai.RateLimitError as e:
        print(f"Error: Rate limit exceeded. {e}")
        return {"sentiment": "Error", "emotion": "Error"}
    except openai.APIStatusError as e:
        print(f"Error: OpenAI API returned an error (Status code: {e.status_code}). {e}")
        return {"sentiment": "Error", "emotion": "Error"}
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return {"sentiment": "Error", "emotion": "Error"}

def format_confusion_matrix(cm, labels, title):
    """Formats a confusion matrix into a readable string."""
    matrix_str = f"\n--- Confusion Matrix ({title}) ---\n"
    
    # Create header row (Predicted labels)
    col_width = max(max(len(label) for label in labels), 7) + 2 # Get max label length for padding
    header = "True \\ Pred".ljust(col_width) + " | "
    for label in labels:
        header += f"{label:^{col_width}} | "
    matrix_str += header + "\n"
    matrix_str += "-" * len(header) + "\n"
    
    # Create matrix rows (True labels)
    for i, label in enumerate(labels):
        row_str = f"{label.ljust(col_width)} | "
        for val in cm[i]:
            row_str += f"{str(val):^{col_width}} | "
        matrix_str += row_str + "\n"
    
    return matrix_str + "\n"

def test_model_accuracy(csv_file_path: str, sample_size: int = 50):
    """
    Loads a CSV, runs sentiment/emotion analysis on a sample, and reports accuracy.
    Saves the detailed results to '[model_name]_[timestamp]_results.csv' and
    the classification report to '[model_name]_[timestamp]_report.txt'.
    """
    print(f"Loading dataset from {csv_file_path}...")
    try:
        # Try parsing as a standard CSV first (comma-separated)
        df = pd.read_csv(csv_file_path)
        
    except FileNotFoundError:
        print(f"Error: File not found at '{csv_file_path}'.")
        print("Please make sure the CSV file path is correct.")
        return
    except pd.errors.ParserError as e:
        # This catches errors like 'Error tokenizing data'
        print(f"Standard CSV parsing failed: {e}")
        print("Trying again with tab delimiter...")
        try:
            # Fallback to tab-separated (which the previous file was)
            df = pd.read_csv(csv_file_path, sep='\t')
        except Exception as e2:
            print(f"Tab-parsing failed: {e2}. Trying with 'python' engine...")
            try:
                # Final fallback for complex, messy files
                df = pd.read_csv(csv_file_path, engine='python', on_bad_lines='skip')
            except Exception as e3:
                print(f"Fatal error: Could not parse file even with fallbacks. {e3}")
                return
    except Exception as e:
        # Catch other potential errors (e.g., permissions)
        print(f"General error loading file: {e}")
        return

    # --- IMPORTANT: Adjust these column names based on your CSV ---
    TEXT_COLUMN = "tweet" 
    LABEL_COLUMN = "label"
    # -----------------------------------------------------------
    
    # --- Define the model to use for this test run ---
    # This variable will be used for the API call and the report filenames
    model_name = "gpt-4.1" # <-- CHANGED FROM gpt-5
    # -------------------------------------------------

    if TEXT_COLUMN not in df.columns or LABEL_COLUMN not in df.columns:
        print(f"Error: Columns '{TEXT_COLUMN}' or '{LABEL_COLUMN}' not found in the file.")
        print(f"Available columns are: {list(df.columns)}")
        print("This might happen if the delimiter is still incorrect or the file is corrupt.")
        return

    # Limit to a sample to avoid high API costs/time
    if len(df) > sample_size:
        print(f"Using a random sample of {sample_size} tweets from {len(df)} total.")
        df_sample = df.sample(n=sample_size, random_state=42) # random_state for reproducible results
    else:
        print(f"Using all {len(df)} tweets in the file.")
        df_sample = df

    print(f"\n--- Starting Sentiment Analysis ---")
    print(f"Model: {model_name}")
    print(f"WARNING: This will make {len(df_sample)} API calls to OpenAI.")
    print("This may take some time and incur costs.\n")
    
    results_list = [] # Store detailed results for saving to CSV

    # Map Indonesian labels from CSV to the model's English output
    # This is an EMOTION dataset, so we map emotions to sentiment.
    label_mapping = {
        "love": "Positive",
        "happy": "Positive",\
        "anger": "Negative",
        "sadness": "Negative",
        "fear": "Negative"
        # We ignore any other labels as they don't map to sentiment.
    }

    processed_count = 0
    for index, row in df_sample.iterrows():
        text = str(row[TEXT_COLUMN])
        true_label_raw = str(row[LABEL_COLUMN]).strip() # This is a string (e.g., "love")
        
        # --- Skip rows with empty text or labels ---
        if not text or pd.isna(text):
            print(f"Skipping row {index}: Empty text.")
            continue
        if pd.isna(true_label_raw) or not true_label_raw:
            print(f"Skipping row {index}: Empty label.")
            continue
            
        # Get the mapped sentiment (e.g., "Positive") from the emotion label (e.g., "love")
        true_sentiment = label_mapping.get(true_label_raw)
        
        if true_sentiment is None:
            print(f"Skipping row {index}: Unknown/unmapped true label '{true_label_raw}'.")
            continue
        # ---------------------------------------------

        # Call the API to get the predicted sentiment and emotion
        # Pass the defined model_name to the function
        prediction_data = get_sentiment(text, model_name)
        
        predicted_sentiment = prediction_data.get("sentiment", "Unknown")
        predicted_emotion = prediction_data.get("emotion", "Unknown")
        
        processed_count += 1
        
        # --- Store results for CSV export ---
        is_correct_sentiment = False
        is_correct_emotion = False
        if predicted_sentiment not in ["Error", "Unknown"]:
            is_correct_sentiment = (true_sentiment == predicted_sentiment)
            is_correct_emotion = (true_label_raw == predicted_emotion)
        
        results_list.append({
            "Text": text,
            "True_Emotion": true_label_raw, 
            "True_Sentiment": true_sentiment,
            "Predicted_Emotion": predicted_emotion,
            "Predicted_Sentiment": predicted_sentiment,
            "Correct_Sentiment": is_correct_sentiment,
            "Correct_Emotion": is_correct_emotion
        })
        # -----------------------------------

        if predicted_sentiment not in ["Error", "Unknown"]:
            print(f"\nTweet Text: {text}")
            print(f"Processed tweet {processed_count}/{len(df_sample)}...")
            print(f"  > Sentiment: True: {true_sentiment}, Pred: {predicted_sentiment}")
            print(f"  > Emotion:   True: {true_label_raw}, Pred: {predicted_emotion}")
        else:
            print(f"\nTweet Text: {text}") # Also print text on error/skip
            print(f"Skipping tweet {processed_count}/{len(df_sample)} due to API error: {predicted_sentiment}")

        # IMPORTANT: Add a small delay to avoid hitting API rate limits
        time.sleep(0.5) # 0.5 second delay between requests

    print("\n--- Analysis Complete ---")
    
    # --- Save detailed results to CSV ---
    if not results_list:
        print("No results to report. All rows may have been skipped.")
        return
        
    results_df = pd.DataFrame(results_list)
    
    # --- Generate dynamic file paths ---
    # The model_name variable is now correctly used here
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    results_csv_path = f"{model_name}_{timestamp}_results.csv"
    # -----------------------------------
    
    try:
        results_df.to_csv(results_csv_path, index=False, encoding='utf-8-sig')
        print(f"\nSaved detailed results to {results_csv_path}")
    except Exception as e:
        print(f"\nError saving results CSV: {e}")
    # ----------------------------------

    # --- Calculate metrics from valid predictions ---
    valid_results_df = results_df[~results_df['Predicted_Sentiment'].isin(["Error", "Unknown"])]
    
    if valid_results_df.empty:
        print("No valid predictions were made. Cannot generate report.")
        return
        
    # --- 1. SENTIMENT Report ---
    y_true_sentiment = valid_results_df['True_Sentiment']
    y_pred_sentiment = valid_results_df['Predicted_Sentiment']
    all_labels_sentiment = ["Positive", "Negative", "Neutral"]

    accuracy_sentiment = accuracy_score(y_true_sentiment, y_pred_sentiment)
    report_sentiment = classification_report(
        y_true_sentiment, 
        y_pred_sentiment, 
        labels=all_labels_sentiment, 
        zero_division=0
    )
    
    # --- Create Sentiment Confusion Matrix ---
    cm_sentiment = confusion_matrix(y_true_sentiment, y_pred_sentiment, labels=all_labels_sentiment)
    cm_str_sentiment = format_confusion_matrix(cm_sentiment, all_labels_sentiment, "Sentiment")
    
    # --- 2. EMOTION Report ---
    y_true_emotion = valid_results_df['True_Emotion']
    y_pred_emotion = valid_results_df['Predicted_Emotion']
    # Get all unique emotion labels present in the true data for the report
    all_labels_emotion = sorted(list(label_mapping.keys()))

    accuracy_emotion = accuracy_score(y_true_emotion, y_pred_emotion)
    report_emotion = classification_report(
        y_true_emotion, 
        y_pred_emotion, 
        labels=all_labels_emotion, 
        zero_division=0
    )
    
    # --- Create Emotion Confusion Matrix ---
    cm_emotion = confusion_matrix(y_true_emotion, y_pred_emotion, labels=all_labels_emotion)
    cm_str_emotion = format_confusion_matrix(cm_emotion, all_labels_emotion, "Emotion")

    
    # --- Print to Console ---
    print("\n--- SENTIMENT ANALYSIS REPORT ---")
    print(f"Accuracy: {accuracy_sentiment * 100:.2f}%\n")
    print("Classification Report (Sentiment):")
    print(report_sentiment)
    print(cm_str_sentiment) # Print sentiment confusion matrix
    
    print("\n--- EMOTION ANALYSIS REPORT ---")
    print(f"Accuracy: {accuracy_emotion * 100:.2f}%\n")
    print("Classification Report (Emotion):")
    print(report_emotion)
    print(cm_str_emotion) # Print emotion confusion matrix
    
    # --- Save classification report to TXT ---
    # Use the same model and timestamp from above
    report_txt_path = f"{model_name}_{timestamp}_report.txt"
    
    try:
        with open(report_txt_path, "w", encoding="utf-8") as f:
            f.write(f"Sentiment & Emotion Analysis Report\n")
            f.write(f"Source File: {csv_file_path}\n")
            f.write(f"Sample Size: {sample_size} (Processed: {len(results_df)}, Valid API Responses: {len(valid_results_df)})\n")
            f.write(f"Model: {model_name}\n") # Use variable
            
            f.write("\n\n--- SENTIMENT ANALYSIS REPORT ---\n")
            f.write("----------------------------------\n")
            f.write(f"Accuracy: {accuracy_sentiment * 100:.2f}%\n\n")
            f.write("Classification Report (Sentiment):")
            f.write(report_sentiment)
            f.write(cm_str_sentiment) # Save sentiment confusion matrix
            
            f.write("\n\n--- EMOTION ANALYSIS REPORT ---\n")
            f.write("----------------------------------\n")
            f.write(f"Accuracy: {accuracy_emotion * 100:.2f}%\n\n")
            f.write("Classification Report (Emotion):")
            f.write(report_emotion)
            f.write(cm_str_emotion) # Save emotion confusion matrix
            
        print(f"\nSaved combined classification reports to {report_txt_path}")
    except Exception as e:
        print(f"\nError saving report TXT: {e}")
    # -----------------------------------------


# --- Example Usage ---
if __name__ == "__main__":
    # --- Comment out old examples ---
    # text1 = "Makanannya enak sekali dan pelayanannya ramah!"
    # sentiment1 = get_sentiment(text1, "gpt-5") # Requires model name now
    # print(f"Text: '{text1}'\nSentiment: {sentiment1}\n")

    # text2 = "This movie was incredibly boring. I fell asleep halfway through."
    # sentiment2 = get_sentiment(text2, "gpt-5") # Requires model name now
    # print(f"Text: '{text2}'\nSentiment: {sentiment2}\n")

    # text3 = "The package is scheduled to arrive tomorrow at 4 PM."
    # sentiment3 = get_sentiment(text3, "gpt-5") # Requires model name now
    # print(f"Text: '{text3}'\nSentiment: {sentiment3}\n")

    # text4 = "The food was great, but the music was way too loud."
    # sentiment4 = get_sentiment(text4, "gpt-5") # Requires model name now
    # print(f"Text: '{text4}'\nSentiment: {sentiment4}\n")

    # --- New Accuracy Testing ---
    # IMPORTANT: Place your CSV file in the same folder as this script
    # Make sure this file name is correct.
    # Updated to the path you provided.
    FILE_PATH = "<YOUR_CSV_FILE_PATH_HERE>"
    
    # Start with a small sample (e.g., 50) to test.
    # Running all 10,000+ rows will be slow and cost money.
    # You can increase this number once you confirm it's working.
    SAMPLE_SIZE = 440
    
    test_model_accuracy(FILE_PATH, SAMPLE_SIZE)

